In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import savgol_filter

# 1. Load file
file_path = 'Trypsin_Actin_MPGAP_Data.xlsx'
df = pd.read_excel(file_path) if file_path.endswith('.xlsx') else pd.read_csv(file_path)

dt = df['Time'].diff().mean()
time_arr = df['Time'].values

# Automatic detection of ROI pairs
roi_pairs = [(c.replace('_Actin', ''), c, f"{c.replace('_Actin', '')}_MPGAP")
             for c in df.columns if '_Actin' in c and f"{c.replace('_Actin', '')}_MPGAP" in df.columns]

# --- Analysis parameters ---
SMOOTH_WINDOW = 3   # Smoothing window (odd number; adjust to 5, 7, 9, etc., depending on noise levels)
SEARCH_WINDOW = 60  # Peak lag search window (±60 s)

corr_list = []
results_summary = []

for roi_name, actin_col, rhoa_col in roi_pairs:
    actin_raw = df[actin_col].values
    rhoa_raw = df[rhoa_col].values

    # Denoising using Savitzky-Golay filter
    actin_sm = savgol_filter(actin_raw, window_length=SMOOTH_WINDOW, polyorder=2)
    rhoa_sm = savgol_filter(rhoa_raw, window_length=SMOOTH_WINDOW, polyorder=2)

    # Z-score normalization (normalizes amplitude differences to compare signal patterns directly)
    act_norm = (actin_sm - np.mean(actin_sm)) / np.std(actin_sm)
    rho_norm = (rhoa_sm - np.mean(rhoa_sm)) / np.std(rhoa_sm)

    # Calculate cross-correlation
    corr = signal.correlate(rho_norm, act_norm, mode='full') / len(act_norm)
    lags = signal.correlation_lags(len(act_norm), len(rho_norm), mode='full')
    time_lags = lags * dt

    # Search for peak with maximum correlation coefficient within search window (-60 s to +60 s)
    valid_mask = (time_lags >= -SEARCH_WINDOW) & (time_lags <= SEARCH_WINDOW)
    valid_lags = time_lags[valid_mask]
    valid_corr = corr[valid_mask]

    max_idx = np.argmax(valid_corr)
    peak_lag = valid_lags[max_idx]      # Time lag corresponding to max correlation (Phase Lag)
    max_corr = valid_corr[max_idx]      # Maximum correlation coefficient (r)

    corr_list.append(corr)
    results_summary.append({
        'ROI': roi_name,
        'Phase Lag (s)': peak_lag,
        'Max Correlation (r)': max_corr
    })

# Combine into summary DataFrame
summary_df = pd.DataFrame(results_summary)

# Calculate mean cross-correlation waveform across all ROIs
corr_matrix = np.array(corr_list)
mean_corr = np.mean(corr_matrix, axis=0)

valid_mask_mean = (time_lags >= -SEARCH_WINDOW) & (time_lags <= SEARCH_WINDOW)
valid_lags_mean = time_lags[valid_mask_mean]
valid_mean_corr = mean_corr[valid_mask_mean]

overall_max_idx = np.argmax(valid_mean_corr)
overall_peak_lag = valid_lags_mean[overall_max_idx]
overall_max_corr = valid_mean_corr[overall_max_idx]

# Output results
print("=== Intensity Cross-Correlation Analysis Results ===")
print(summary_df.to_string(index=False))

mean_lag = summary_df['Phase Lag (s)'].mean()
sem_lag = summary_df['Phase Lag (s)'].std() / np.sqrt(len(summary_df))
mean_r = summary_df['Max Correlation (r)'].mean()

print("\n--- Statistical Results ---")
print(f"All ROIs Mean Phase Lag: {mean_lag:.2f} ± {sem_lag:.2f} s")
print(f"All ROIs Mean Max Correlation (r): {mean_r:.3f}")
print(f"Overall Peak Lag from mean cross-correlation curve: {overall_peak_lag:.2f} s (Max r = {overall_max_corr:.3f})")

# --- Plotting graphs ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=300)

# [Spine and tick line width settings]
FRAME_WIDTH = 1.5  # Frame linewidth

for ax in axes:
    # Make plot frame spines (top, bottom, left, right) thicker
    for spine in ax.spines.values():
        spine.set_linewidth(FRAME_WIDTH)
    # Make axis tick lines thicker to match the frame
    ax.tick_params(width=FRAME_WIDTH)

# (1) Cross-correlation curves for individual ROIs and mean
for i, corr in enumerate(corr_list):
    axes[0].plot(time_lags, corr, color='gray', alpha=0.4, linewidth=1,
                 label='Individual ROI' if i == 0 else "")
axes[0].plot(time_lags, mean_corr, color='blue', linewidth=2.5, label='Mean Cross-Correlation')
axes[0].axvline(x=overall_peak_lag, color='red', linestyle='--', label=f'Peak Lag = {overall_peak_lag:.1f} s')
axes[0].axhline(y=0, color='black', linestyle=':', alpha=0.5)
axes[0].set_xlim(-SEARCH_WINDOW, SEARCH_WINDOW)
axes[0].set_xlabel('Time Lag (s)\n(Positive = MPGAP lags behind Actin)')
axes[0].set_ylabel('Cross-Correlation Coefficient (r)')
axes[0].set_title('A. Cross-Correlation Curves (Intensity)')

# --- Position legend outside ---
axes[0].legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

# Disable grid lines
axes[0].grid(False)

# (2) Distribution of Phase Lag (boxplot)
axes[1].boxplot(summary_df['Phase Lag (s)'], positions=[1], patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7, linewidth=FRAME_WIDTH),
                medianprops=dict(linewidth=FRAME_WIDTH),
                whiskerprops=dict(linewidth=FRAME_WIDTH),
                capprops=dict(linewidth=FRAME_WIDTH))
axes[1].scatter(np.ones(len(summary_df)), summary_df['Phase Lag (s)'], color='navy', zorder=3)
axes[1].axhline(y=0, color='gray', linestyle=':')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['ROIs'])
axes[1].set_ylabel('Phase Lag (s)')
axes[1].set_title(f'B. Phase Lag Distribution\n(Mean: {mean_lag:.1f} ± {sem_lag:.1f} s)')

# Disable grid lines
axes[1].grid(False)

# Adjust layout so legends and axis labels do not clip
plt.tight_layout()
plt.show()